In [0]:
from pyspark.sql.functions import *

catalog = "bike_data"
bronze_schema = "bronze"
silver_schema = "silver"

print("=" * 80)
print("SILVER TRANSFORMATION: crm_sales_details")
print("=" * 80)

# Section 1: Read Bronze table
print("\nSection 1: Reading Bronze table")
df = spark.table(f"{catalog}.{bronze_schema}.crm_sales_details")
initial_count = df.count()
print(f"Initial rows: {initial_count}")

# Section 2: Transform data
print("\nSection 2: Data transformation")

# 2.1 Remove duplicates
print("  2.1 Removing duplicates by sls_ord_num")
df = df.dropDuplicates(subset=["sls_ord_num"])
print(f"  Rows after dedup: {df.count()}")

# 2.2 Fix dates (容错多种格式，包括YYYYMMDD)
print("  2.2 Fixing date columns")
df = df.withColumn("sls_order_dt",
    coalesce(
        try_to_date(col("sls_order_dt"), "yyyy-MM-dd"),
        try_to_date(col("sls_order_dt"), "MM/dd/yyyy"),
        try_to_date(col("sls_order_dt"), "yyyyMMdd"),
        try_to_date(col("sls_order_dt"), "dd-MM-yyyy")
    )
)

df = df.withColumn("sls_ship_dt",
    coalesce(
        try_to_date(col("sls_ship_dt"), "yyyy-MM-dd"),
        try_to_date(col("sls_ship_dt"), "MM/dd/yyyy"),
        try_to_date(col("sls_ship_dt"), "yyyyMMdd"),
        try_to_date(col("sls_ship_dt"), "dd-MM-yyyy")
    )
)

df = df.withColumn("sls_due_dt",
    coalesce(
        try_to_date(col("sls_due_dt"), "yyyy-MM-dd"),
        try_to_date(col("sls_due_dt"), "MM/dd/yyyy"),
        try_to_date(col("sls_due_dt"), "yyyyMMdd"),
        try_to_date(col("sls_due_dt"), "dd-MM-yyyy")
    )
)

# 2.3 Validate numeric
print("  2.3 Validating numeric columns")
df = df.withColumn("sls_quantity",
    when(col("sls_quantity") > 0, col("sls_quantity"))
    .otherwise(None)
)
df = df.withColumn("sls_price",
    when(col("sls_price") > 0, col("sls_price"))
    .otherwise(None)
)
df = df.withColumn("sls_sales",
    when(col("sls_sales") > 0, col("sls_sales"))
    .otherwise(None)
)

# 2.4 Remove invalid rows
print("  2.4 Removing invalid rows")
df = df.filter(col("sls_cust_id").isNotNull())
df = df.filter(col("sls_prd_key").isNotNull())

print(f"  Rows after transformations: {df.count()}")

# Section 3: Sanity checks
print("\nSection 3: Sanity checks")
null_check = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
display(null_check)
display(df.limit(3))

# Section 4: Write to Silver
print("\nSection 4: Writing to Silver table")
silver_table = f"{catalog}.{silver_schema}.sales"
df.write.mode("overwrite").format("delta").saveAsTable(silver_table)

final_count = df.count()
print(f"Written to: {silver_table}")
print(f"Final rows: {final_count}")
print(f"Removed: {initial_count - final_count} rows")